<a href="https://colab.research.google.com/github/SanyaKapoor/Double-ML-for-Impact-Assessment/blob/main/Dataframe_Construction_for_running_Double_ML_analysis_(all_seasons).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installations




In [ ]:
!pip install shap
!pip3 install psmpy

#Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import math
from psmpy import PsmPy
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
from datetime import date
import ee
import shap
from google.colab import drive
import warnings
from datetime import date
from sklearn.model_selection import train_test_split

#Initialisations

In [ ]:
#Earth Engine
ee.Authenticate(force=True)
ee.Initialize(project='ee-sanyakapoor')

#Shap
shap.initjs()

#Drive
drive.mount('/content/drive', force_remount=True)
warnings.filterwarnings("ignore")

#Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 15)

Mounted at /content/drive


#Dataframe Configurations

In [ ]:
#Setting the current date
today = date.today()
str_date = today.strftime("%b-%d-%Y")

#Set the season here - 1 (Kharif), 2 (Rabi), 3 (Zaid)
season = 1
if season == 1:
    season_name = "Kharif"
elif season == 2:
    season_name = "Rabi"
elif season == 3:
    season_name = "Zaid"
else:
    raise Exception("Please set season value")

#Set frequency threshold for deciding bianry values of drought later
freq_threshold = 5

#Set the dataset to be imported here - 1 (built in 2017), 2 (valid farm ponds)
dataset = 2

#PSM Matching - If 0, then #cp = 3*#tp
psm_matched = 0

#Importing static covariate data (for example - slope)

In [ ]:
#File location
%cd /content/drive/MyDrive/Anika_Murarka/AEZ10/

#File name
csvfile = 'AEZ_10_static_covariates_all.csv'

#Loading the file to a pandas dataframe
df_cp = pd.read_csv(csvfile)

/content/drive/MyDrive/Anika_Murarka/AEZ10


In [ ]:
#Visualising the data of control sites
df_cp.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,system:index,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,recent_cropping_intensity,slope,.geo,HSG,pH,OC,CEC,drainage_density
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0000,0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,1.0,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974
1,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0000,1,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,2.0,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279
2,3,17003565236,minakshi talab nanaji/shivlal,BARGHAT,SEONI,MADHYA PRADESH,393524.7500,2,1,67.943964,415.737145,2427.151790,217.142019,0.000000,535.673841,4,21.910636,79.769193,6.293255,3.0,2.446378,"{""type"":""Point"",""coordinates"":[79.769193,21.91...",4.0,7.9,0.75,43.0,0.005631
3,4,17003503762,Talab Gaharikarna work,BARGHAT,SEONI,MADHYA PRADESH,891920.0000,3,1,107.416322,749.393619,3577.476312,12.767860,19.215269,546.366545,1,21.944641,79.819831,12.000000,1.0,2.098391,"{""type"":""Point"",""coordinates"":[79.819831,21.94...",4.0,7.9,0.75,43.0,0.004829
4,5,17002967296,Minakshi talab Sabulal Bairagi,BARGHAT,SEONI,MADHYA PRADESH,345572.1775,4,1,176.630941,1181.717424,2239.538528,132.323610,0.000000,537.105898,4,21.949203,79.836798,3.888563,3.0,3.709937,"{""type"":""Point"",""coordinates"":[79.836798,21.94...",4.0,7.9,0.75,43.0,0.005205


In [ ]:
# #Here, one of the 2 files will be selected based on the option selected above
# if dataset == 1:
#     csvfile = 'AEZ_13_static_covariates_built_in_2017_v2.csv'
# else:
#     csvfile = 'AEZ_13_static_covariates_valid_farm_ponds_v2.csv'

# #Loading selected file to a dataframe
# df_data = pd.read_csv(csvfile)

Check with maam if this change is okay - they had done this to understand difference in RQs [RQ3 - valid farm ponds]

In [ ]:
df_data = df_cp[df_cp['Treatment'] == 1]

In [ ]:
df_cp = df_cp[df_cp['Treatment'] == 0]

In [ ]:
#Removing the unnamed index column
df_data = df_data.drop(['Unnamed: 0'], axis=1, errors='ignore')
df_cp = df_cp.drop(['Unnamed: 0'], axis=1, errors='ignore')

In [ ]:
#Removing the unnamed index column
df_data = df_data.drop(['system:index'], axis=1, errors='ignore')
df_cp = df_cp.drop(['system:index'], axis=1, errors='ignore')

In [ ]:
df_data.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,recent_cropping_intensity,slope,.geo,HSG,pH,OC,CEC,drainage_density
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0000,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,1.0,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974
1,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0000,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,2.0,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279
2,3,17003565236,minakshi talab nanaji/shivlal,BARGHAT,SEONI,MADHYA PRADESH,393524.7500,1,67.943964,415.737145,2427.151790,217.142019,0.000000,535.673841,4,21.910636,79.769193,6.293255,3.0,2.446378,"{""type"":""Point"",""coordinates"":[79.769193,21.91...",4.0,7.9,0.75,43.0,0.005631
3,4,17003503762,Talab Gaharikarna work,BARGHAT,SEONI,MADHYA PRADESH,891920.0000,1,107.416322,749.393619,3577.476312,12.767860,19.215269,546.366545,1,21.944641,79.819831,12.000000,1.0,2.098391,"{""type"":""Point"",""coordinates"":[79.819831,21.94...",4.0,7.9,0.75,43.0,0.004829
4,5,17002967296,Minakshi talab Sabulal Bairagi,BARGHAT,SEONI,MADHYA PRADESH,345572.1775,1,176.630941,1181.717424,2239.538528,132.323610,0.000000,537.105898,4,21.949203,79.836798,3.888563,3.0,3.709937,"{""type"":""Point"",""coordinates"":[79.836798,21.94...",4.0,7.9,0.75,43.0,0.005205


In [ ]:
#The shape of all valid farm ponds vs control points
print(df_cp.shape)
print(df_data.shape)

(4500, 26)
(1500, 26)


#Separating 20% treated point rows: for use later

In [ ]:
_, df_reserved_data = train_test_split(df_data, test_size=0.2, random_state=42)

In [ ]:
print(df_reserved_data.shape)
print(df_data.shape)

(300, 26)
(1500, 26)


#Combining farm ponds and control sites data into a single dataframe

In [ ]:
#Ensuring that the dimensions are same
print(set(df_data.columns) - set(df_cp.columns))

set()


In [ ]:
#Combine control points and 80% of valid farm points
df =  pd.concat([df_data, df_cp])
df.shape

(6000, 26)

#Adding drought indicators

In [ ]:
# Merging UID based drought Indicators
csvfile = 'AEZ10_drought_indicators_sampled_points_global.csv'
df_drought_data = pd.read_csv(csvfile)
df_drought_data = df_drought_data.drop(['Unnamed: 0', '.geo', 'system:index'], axis=1, errors='ignore')
df_data = pd.merge(df_data, df_drought_data, on='UID')

In [ ]:
df_cp = pd.merge(df_cp, df_drought_data, on='UID')

In [ ]:
df_drought_data.head()

,UID,freq_of_drought_2016_at_threshold_1,freq_of_drought_2016_at_threshold_2,freq_of_drought_2016_at_threshold_3,intensity_of_drought_2016_at_threshold_1,intensity_of_drought_2016_at_threshold_2,intensity_of_drought_2016_at_threshold_3,freq_of_drought_2017_at_threshold_1,freq_of_drought_2017_at_threshold_2,freq_of_drought_2017_at_threshold_3,intensity_of_drought_2017_at_threshold_1,intensity_of_drought_2017_at_threshold_2,intensity_of_drought_2017_at_threshold_3,freq_of_drought_2018_at_threshold_1,freq_of_drought_2018_at_threshold_2,freq_of_drought_2018_at_threshold_3,intensity_of_drought_2018_at_threshold_1,intensity_of_drought_2018_at_threshold_2,intensity_of_drought_2018_at_threshold_3,freq_of_drought_2019_at_threshold_1,freq_of_drought_2019_at_threshold_2,freq_of_drought_2019_at_threshold_3,intensity_of_drought_2019_at_threshold_1,intensity_of_drought_2019_at_threshold_2,intensity_of_drought_2019_at_threshold_3,freq_of_drought_2020_at_threshold_1,freq_of_drought_2020_at_threshold_2,freq_of_drought_2020_at_threshold_3,intensity_of_drought_2020_at_threshold_1,intensity_of_drought_2020_at_threshold_2,intensity_of_drought_2020_at_threshold_3,freq_of_drought_2021_at_threshold_1,freq_of_drought_2021_at_threshold_2,freq_of_drought_2021_at_threshold_3,intensity_of_drought_2021_at_threshold_1,intensity_of_drought_2021_at_threshold_2,intensity_of_drought_2021_at_threshold_3,freq_of_drought_2022_at_threshold_1,freq_of_drought_2022_at_threshold_2,freq_of_drought_2022_at_threshold_3,intensity_of_drought_2022_at_threshold_1,intensity_of_drought_2022_at_threshold_2,intensity_of_drought_2022_at_threshold_3
0,1,20,3,0,1.15,2.0,0.0,19,2,0,1.105263,2.0,0.0,21,5,0,1.238095,2.0,0.0,17,0,0,1.0,0.0,0.0,19,0,0,1.000000,0.0,0.0,21,4,0,1.190476,2.0,0.0,19,1,0,1.052632,2.0,0.0
1,4,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,18,0,0,1.0,0.0,0.0,17,4,0,1.235294,2.0,0.0,20,0,0,1.000000,0.0,0.0,18,0,0,1.000000,0.0,0.0
2,9,20,3,0,1.15,2.0,0.0,18,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,21,0,0,1.0,0.0,0.0,18,4,0,1.222222,2.0,0.0,20,4,0,1.200000,2.0,0.0,18,0,0,1.000000,0.0,0.0
3,19,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,18,0,0,1.0,0.0,0.0,17,4,0,1.235294,2.0,0.0,20,0,0,1.000000,0.0,0.0,18,0,0,1.000000,0.0,0.0
4,22,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,18,0,0,1.0,0.0,0.0,17,4,0,1.235294,2.0,0.0,20,0,0,1.000000,0.0,0.0,18,0,0,1.000000,0.0,0.0


In [ ]:
df =  pd.concat([df_data, df_cp])
df.shape

(6828, 68)

In [ ]:
df.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,recent_cropping_intensity,slope,.geo,HSG,pH,OC,CEC,drainage_density,freq_of_drought_2016_at_threshold_1,freq_of_drought_2016_at_threshold_2,freq_of_drought_2016_at_threshold_3,intensity_of_drought_2016_at_threshold_1,intensity_of_drought_2016_at_threshold_2,intensity_of_drought_2016_at_threshold_3,freq_of_drought_2017_at_threshold_1,freq_of_drought_2017_at_threshold_2,freq_of_drought_2017_at_threshold_3,intensity_of_drought_2017_at_threshold_1,intensity_of_drought_2017_at_threshold_2,intensity_of_drought_2017_at_threshold_3,freq_of_drought_2018_at_threshold_1,freq_of_drought_2018_at_threshold_2,freq_of_drought_2018_at_threshold_3,intensity_of_drought_2018_at_threshold_1,intensity_of_drought_2018_at_threshold_2,intensity_of_drought_2018_at_threshold_3,freq_of_drought_2019_at_threshold_1,freq_of_drought_2019_at_threshold_2,freq_of_drought_2019_at_threshold_3,intensity_of_drought_2019_at_threshold_1,intensity_of_drought_2019_at_threshold_2,intensity_of_drought_2019_at_threshold_3,freq_of_drought_2020_at_threshold_1,freq_of_drought_2020_at_threshold_2,freq_of_drought_2020_at_threshold_3,intensity_of_drought_2020_at_threshold_1,intensity_of_drought_2020_at_threshold_2,intensity_of_drought_2020_at_threshold_3,freq_of_drought_2021_at_threshold_1,freq_of_drought_2021_at_threshold_2,freq_of_drought_2021_at_threshold_3,intensity_of_drought_2021_at_threshold_1,intensity_of_drought_2021_at_threshold_2,intensity_of_drought_2021_at_threshold_3,freq_of_drought_2022_at_threshold_1,freq_of_drought_2022_at_threshold_2,freq_of_drought_2022_at_threshold_3,intensity_of_drought_2022_at_threshold_1,intensity_of_drought_2022_at_threshold_2,intensity_of_drought_2022_at_threshold_3
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0000,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,1.0,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,20,3,0,1.15,2.0,0.0,19,2,0,1.105263,2.0,0.0,21,5,0,1.238095,2.0,0.0,17,0,0,1.0,0.0,0.0,19,0,0,1.000000,0.0,0.0,21,4,0,1.190476,2.0,0.0,19,1,0,1.052632,2.0,0.0
1,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0000,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,2.0,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279,20,3,0,1.15,2.0,0.0,19,2,0,1.105263,2.0,0.0,21,5,0,1.238095,2.0,0.0,17,0,0,1.0,0.0,0.0,19,0,0,1.000000,0.0,0.0,21,4,0,1.190476,2.0,0.0,19,1,0,1.052632,2.0,0.0
2,3,17003565236,minakshi talab nanaji/shivlal,BARGHAT,SEONI,MADHYA PRADESH,393524.7500,1,67.943964,415.737145,2427.151790,217.142019,0.000000,535.673841,4,21.910636,79.769193,6.293255,3.0,2.446378,"{""type"":""Point"",""coordinates"":[79.769193,21.91...",4.0,7.9,0.75,43.0,0.005631,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,18,0,0,1.0,0.0,0.0,17,4,0,1.235294,2.0,0.0,20,0,0,1.000000,0.0,0.0,18,0,0,1.000000,0.0,0.0
3,4,17003503762,Talab Gaharikarna work,BARGHAT,SEONI,MADHYA PRADESH,891920.0000,1,107.416322,749.393619,3577.476312,12.767860,19.215269,546.366545,1,21.944641,79.819831,12.000000,1.0,2.098391,"{""type"":""Point"",""coordinates"":[79.819831,21.94...",4.0,7.9,0.75,43.0,0.004829,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2.0,0.0,18,0,0,1.0,0.0,0.0,17,4,0,1.235294,2.0,0.0,20,0,0,1.000000,0.0,0.0,18,0,0,1.000000,0.0,0.0
4,5,17002967296,Minakshi talab Sabulal Bairagi,BARGHAT,SEONI,MADHYA PRADESH,345572.1775,1,176.630941,1181.717424,2239.538528,132.323610,0.000000,537.105898,4,21.949203,79.836798,3.888563,3.0,3.709937,"{""type"":""Point"",""coordinates"":[79.836798,21.94...",4.0,7.9,0.75,43.0,0.005205,20,0,0,1.00,0.0,0.0,20,0,0,1.000000,0.0,0.0,21,5,0,1.238095,2

In [ ]:
df = df.drop_duplicates()
df.shape

(5606, 68)

In [ ]:
df[df['Treatment'] == 0].shape

(4106, 68)

We can ignore if control is not exactly equal to thrice treated. like in this case 4500 -> 4100

##Creating observations from these drought indicators

In [ ]:
df_global_all_data = df.copy()
df_global_all_data.shape

(5606, 68)

In [ ]:
df_drought = df.copy()
drop_list = []

for year in range(2016, 2023):
    freq = f'freq_of_drought_{year}_at_threshold_1'
    inten = f'intensity_of_drought_{year}_at_threshold_1'
    drop_list.append(freq)
    drop_list.append(inten)

    freq = f'freq_of_drought_{year}_at_threshold_3'
    inten = f'intensity_of_drought_{year}_at_threshold_3'
    drop_list.append(freq)
    drop_list.append(inten)

df_drought = df_drought.drop(drop_list, axis=1, errors='ignore')
df_drought.shape

for year in range(2016, 2023):
    df_drought.rename(columns={f'freq_of_drought_{year}_at_threshold_2': f'frequency_{year}',
                                  f'intensity_of_drought_{year}_at_threshold_2': f'intensity_{year}'}, inplace=True)

df_drought = df_drought.drop(['Unnamed: 0'], axis=1, errors='ignore')
df_drought.shape

(5606, 40)

In [ ]:
df_drought.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,recent_cropping_intensity,slope,.geo,HSG,pH,OC,CEC,drainage_density,frequency_2016,intensity_2016,frequency_2017,intensity_2017,frequency_2018,intensity_2018,frequency_2019,intensity_2019,frequency_2020,intensity_2020,frequency_2021,intensity_2021,frequency_2022,intensity_2022
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0000,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,1.0,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,3,2.0,2,2.0,5,2.0,0,0.0,0,0.0,4,2.0,1,2.0
1,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0000,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,2.0,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279,3,2.0,2,2.0,5,2.0,0,0.0,0,0.0,4,2.0,1,2.0
2,3,17003565236,minakshi talab nanaji/shivlal,BARGHAT,SEONI,MADHYA PRADESH,393524.7500,1,67.943964,415.737145,2427.151790,217.142019,0.000000,535.673841,4,21.910636,79.769193,6.293255,3.0,2.446378,"{""type"":""Point"",""coordinates"":[79.769193,21.91...",4.0,7.9,0.75,43.0,0.005631,0,0.0,0,0.0,5,2.0,0,0.0,4,2.0,0,0.0,0,0.0
3,4,17003503762,Talab Gaharikarna work,BARGHAT,SEONI,MADHYA PRADESH,891920.0000,1,107.416322,749.393619,3577.476312,12.767860,19.215269,546.366545,1,21.944641,79.819831,12.000000,1.0,2.098391,"{""type"":""Point"",""coordinates"":[79.819831,21.94...",4.0,7.9,0.75,43.0,0.004829,0,0.0,0,0.0,5,2.0,0,0.0,4,2.0,0,0.0,0,0.0
4,5,17002967296,Minakshi talab Sabulal Bairagi,BARGHAT,SEONI,MADHYA PRADESH,345572.1775,1,176.630941,1181.717424,2239.538528,132.323610,0.000000,537.105898,4,21.949203,79.836798,3.888563,3.0,3.709937,"{""type"":""Point"",""coordinates"":[79.836798,21.94...",4.0,7.9,0.75,43.0,0.005205,0,0.0,0,0.0,5,2.0,0,0.0,4,2.0,4,2.0,0,0.0


In [ ]:
retain_cols = [col for col in df.columns if 'frequency' not in col and 'intensity' not in col and 'drought' not in col] + ['recent_cropping_intensity']
new_cols = ['base_year', 'target_year']
df_base_target = pd.DataFrame(columns = retain_cols + new_cols)
df_base_target.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,.geo,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year


I have added .iloc[0] - dict['frequency'] = int(df_drought.loc[df_drought['UID'] == row['UID']][f'frequency_{target_year}'])
.iloc[0]

TO BE DONE - NEED to first confirm that we have data for other years then adjust the loop to have 6 target years

In [ ]:
# 2B:   takes approx 8 minutes
for index, row in df.iterrows():
  dict = {key: row[key] for key in retain_cols}
  for target_year in range(2018, 2022):
    dict['base_year'] = str(2016)
    dict['target_year'] = str(target_year)
    dict['frequency'] = int(df_drought.loc[df_drought['UID'] == row['UID'], f'frequency_{target_year}'].iloc[0])
    dict['intensity'] = float(df_drought.loc[df_drought['UID'] == row['UID']][f'intensity_{target_year}'].iloc[0])
    if dict['frequency'] >= freq_threshold:
        dict['drought'] = 1
    else:
        dict['drought'] = 0
    df_append = pd.DataFrame([dict])
    df_base_target = pd.concat([df_base_target, df_append], ignore_index=True)

freq intensity -> is for target year here

- Why was this done? Why NOT base year? - We do not need base year. We need observations wrt target only

In [ ]:
df_base_target.shape

(22424, 31)

The cell below is imp, It will fix multiple entries for same base_target UID. Drop duplicates is unable to.

In [ ]:
df_base_target = df_base_target.drop_duplicates(subset=['UID', 'base_year', 'target_year'])

In [ ]:
print(f"Unique UIDs in df: {df['UID'].nunique()}")


Unique UIDs in df: 5586


In [ ]:
df_counts = df_base_target.groupby("UID").size()
print(df_counts.value_counts())  # Shows how many rows each UID has


4    5586
Name: count, dtype: int64


##Saving progress

In [ ]:
# %cd /content/drive/MyDrive/Acads/MTP/Question2a/Data/exported_dataframes/
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10/
df_base_target.to_csv(f'DML_Allyears_df_base_target_{season_name}_{str_date}.csv', index=False)
df_final = pd.read_csv(f'DML_Allyears_df_base_target_{season_name}_{str_date}.csv')
df_final.shape
original_cols = [col for col in df_final.columns]

/content/drive/MyDrive/Sanya/DML/AEZ-10


In [ ]:
df_final.shape

(22344, 31)

#Adding NDVI, NDMI, GCI

In [ ]:
#Reading data for cp and valid farm ponds, in this case:
%cd /content/drive/MyDrive/Anika_Murarka/AEZ10/
df_ndvi = pd.read_csv('AEZ10_NDVI_unionmask_all_points.csv')

/content/drive/MyDrive/Anika_Murarka/AEZ10


In [ ]:
df_ndvi.shape

(6000, 79)

In [ ]:
df_ndvi = df_ndvi.drop_duplicates()
df_ndvi.shape

(6000, 79)

In [ ]:
#Add to the main data frame
df_main = pd.merge(df_final, df_ndvi, on='UID')

In [ ]:
df_main.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment_x,dist_closest_crop_x,dist_closest_lin_x,dist_closest_river_x,dist_closest_road_x,dist_closest_upstream_forest_x,elevation_x,flow_accumulation_x,lat,lon,proximity_water_x,slope_x,.geo_x,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity_x,base_year,target_year,frequency,intensity,drought,system:index,GCI_Kharif_2016,GCI_Kharif_2017,GCI_Kharif_2018,GCI_Kharif_2019,GCI_Kharif_2020,GCI_Kharif_2021,GCI_Kharif_2022,GCI_Rabi_2016,GCI_Rabi_2017,GCI_Rabi_2018,GCI_Rabi_2019,GCI_Rabi_2020,GCI_Rabi_2021,GCI_Rabi_2022,GCI_Zaid_2016,GCI_Zaid_2017,GCI_Zaid_2018,GCI_Zaid_2019,GCI_Zaid_2020,GCI_Zaid_2021,GCI_Zaid_2022,Treatment_y,dist_closest_crop_y,dist_closest_lin_y,dist_closest_river_y,dist_closest_road_y,dist_closest_upstream_forest_y,elevation_y,flow_accumulation_y,geo,ndmi_Kharif_2016,ndmi_Kharif_2017,ndmi_Kharif_2018,ndmi_Kharif_2019,ndmi_Kharif_2020,ndmi_Kharif_2021,ndmi_Kharif_2022,ndmi_Rabi_2016,ndmi_Rabi_2017,ndmi_Rabi_2018,ndmi_Rabi_2019,ndmi_Rabi_2020,ndmi_Rabi_2021,ndmi_Rabi_2022,ndmi_Zaid_2016,ndmi_Zaid_2017,ndmi_Zaid_2018,ndmi_Zaid_2019,ndmi_Zaid_2020,ndmi_Zaid_2021,ndmi_Zaid_2022,ndvi_Kharif_2016,ndvi_Kharif_2017,ndvi_Kharif_2018,ndvi_Kharif_2019,ndvi_Kharif_2020,ndvi_Kharif_2021,ndvi_Kharif_2022,ndvi_Rabi_2016,ndvi_Rabi_2017,ndvi_Rabi_2018,ndvi_Rabi_2019,ndvi_Rabi_2020,ndvi_Rabi_2021,ndvi_Rabi_2022,ndvi_Zaid_2016,ndvi_Zaid_2017,ndvi_Zaid_2018,ndvi_Zaid_2019,ndvi_Zaid_2020,ndvi_Zaid_2021,ndvi_Zaid_2022,proximity_water_y,recent_cropping_intensity_y,slope_y,var,.geo_y
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2018,5.0,2.0,1.0,00000000000000000000,0.711542,0.717336,0.809972,0.431295,0.615263,0.888700,0.390608,0.642395,0.658334,0.660641,0.485526,0.593992,0.544453,0.645088,0.554131,0.506600,0.489298,0.436024,0.517683,0.485437,0.548695,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",0.154264,0.129263,0.121871,0.149823,0.152251,0.133397,0.110042,-0.007669,0.009777,-0.004049,0.061286,-0.016264,0.008613,0.017763,-0.075349,-0.079368,-0.079508,-0.039211,-0.064141,-0.021141,-0.060736,0.282017,0.265639,0.307611,0.167375,0.226731,0.330079,0.157530,0.223437,0.228348,0.223971,0.187994,0.200552,0.211346,0.231250,0.166994,0.162020,0.148824,0.150200,0.166941,0.157154,0.177176,0.604106,1.0,3.419834,[<Feature>],"{""type"":""Point"",""coordinates"":[80.454503500374..."
1,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2019,0.0,0.0,0.0,00000000000000000000,0.711542,0.717336,0.809972,0.431295,0.615263,0.888700,0.390608,0.642395,0.658334,0.660641,0.485526,0.593992,0.544453,0.645088,0.554131,0.506600,0.489298,0.436024,0.517683,0.485437,0.548695,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",0.154264,0.129263,0.121871,0.149823,0.152251,0.133397,0.110042,-0.007669,0.009777,-0.004049,0.061286,-0.016264,0.008613,0.017763,-0.075349,-0.079368,-0.079508,-0.039211,-0.064141,-0.021141,-0.060736,0.282017,0.265639,0.307611,0.167375,0.226731,0.330079,0.157530,0.223437,0.228348,0.223971,0.187994,0.200552,0.211346,0.231250,0.166994,0.162020,0.148824,0.150200,0.166941,0.157154,0.177176,0.604106,1.0,3.419834,[<Feature>],"{""type"":""Point"",""coordinates"":[80.454503500374..."
2,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""co

In [ ]:
df_main.shape

(24000, 109)

##Creating observations using NDVI, NDMI, GCI

In [ ]:
conflicting_cols = [col[:-2] for col in df_main.columns if col.endswith('_x') or col.endswith('_y')]
conflicting_cols = list(set(conflicting_cols))  # Remove duplicates
print("Conflicting columns:", conflicting_cols)

Conflicting columns: ['dist_closest_upstream_forest', 'proximity_water', 'elevation', 'Treatment', 'slope', 'flow_accumulation', 'dist_closest_lin', 'recent_cropping_intensity', 'dist_closest_crop', '.geo', 'dist_closest_river', 'dist_closest_road']


In [ ]:
for col in conflicting_cols:
    col_x = col + '_x'
    col_y = col + '_y'
    df_main.rename(columns={col_x: col}, inplace=True)
    df_main.drop(col_y, axis=1, inplace=True)

In [ ]:
df_main_global = df_main.copy()
for index, row in df_main.iterrows():
  df_main.loc[index, f'ndvi_{season_name}_max'] = max(row[f'ndvi_{season_name}_2016'], row[f'ndvi_{season_name}_2018'],row[f'ndvi_{season_name}_2019'], row[f'ndvi_{season_name}_2020'], row[f'ndvi_{season_name}_2021'])
  df_main.loc[index, f'ndvi_{season_name}_min'] = min(row[f'ndvi_{season_name}_2016'], row[f'ndvi_{season_name}_2018'],row[f'ndvi_{season_name}_2019'], row[f'ndvi_{season_name}_2020'], row[f'ndvi_{season_name}_2021'])
for index, row in df_main.iterrows():
  df_main.loc[index, f'ndmi_{season_name}_max'] = max(row[f'ndmi_{season_name}_2016'], row[f'ndmi_{season_name}_2018'],row[f'ndmi_{season_name}_2019'], row[f'ndmi_{season_name}_2020'], row[f'ndmi_{season_name}_2021'])
  df_main.loc[index, f'ndmi_{season_name}_min'] = min(row[f'ndmi_{season_name}_2016'], row[f'ndmi_{season_name}_2018'],row[f'ndmi_{season_name}_2019'], row[f'ndmi_{season_name}_2020'], row[f'ndmi_{season_name}_2021'])
for index, row in df_main.iterrows():
  df_main.loc[index, f'GCI_{season_name}_max'] = max(row[f'GCI_{season_name}_2016'], row[f'GCI_{season_name}_2018'],row[f'GCI_{season_name}_2019'], row[f'GCI_{season_name}_2020'], row[f'GCI_{season_name}_2021'])
  df_main.loc[index, f'GCI_{season_name}_min'] = min(row[f'GCI_{season_name}_2016'], row[f'GCI_{season_name}_2018'],row[f'GCI_{season_name}_2019'], row[f'GCI_{season_name}_2020'], row[f'GCI_{season_name}_2021'])
  new_cols1 = [f'ndvi_{season_name}_target', f'ndvi_{season_name}_target_norm', f'ndvi_{season_name}_base', f'ndvi_{season_name}_base_norm']

new_cols2 = [f'ndmi_{season_name}_target', f'ndmi_{season_name}_target_norm', f'ndmi_{season_name}_base', f'ndmi_{season_name}_base_norm']
new_cols2 = [f'GCI_{season_name}_target', f'GCI_{season_name}_target_norm', f'GCI_{season_name}_base', f'GCI_{season_name}_base_norm']
df_main_ndvi = pd.DataFrame(columns = original_cols + new_cols1 + new_cols2)

TO DO - REMOVE NORMALISED VALUES, NOT NEEDED

In [ ]:
# takes time. Approx 10 mins for 40k rows
for index, row in df_main.iterrows():
  dict = {key: row[key] for key in original_cols}
  base_year = row['base_year']
  target_year = row['target_year']
  dict[f'ndvi_{season_name}_base'] = row[f'ndvi_{season_name}_{base_year}']
  dict[f'ndvi_{season_name}_base_norm'] = (row[f'ndvi_{season_name}_{base_year}'] - row[f'ndvi_{season_name}_min'])/(row[f'ndvi_{season_name}_max'] - row[f'ndvi_{season_name}_min'] + 0.00001)
  dict[f'ndvi_{season_name}_target'] = row[f'ndvi_{season_name}_{target_year}']
  dict[f'ndvi_{season_name}_target_norm'] = (row[f'ndvi_{season_name}_{target_year}'] - row[f'ndvi_{season_name}_min'])/(row[f'ndvi_{season_name}_max'] - row[f'ndvi_{season_name}_min'] + 0.00001)

  dict[f'ndmi_{season_name}_base'] = row[f'ndmi_{season_name}_{base_year}']
  dict[f'ndmi_{season_name}_base_norm'] = (row[f'ndmi_{season_name}_{base_year}'] - row[f'ndmi_{season_name}_min'])/(row[f'ndmi_{season_name}_max'] - row[f'ndmi_{season_name}_min'] + 0.00001)
  dict[f'ndmi_{season_name}_target'] = row[f'ndmi_{season_name}_{target_year}']
  dict[f'ndmi_{season_name}_target_norm'] = (row[f'ndmi_{season_name}_{target_year}'] - row[f'ndmi_{season_name}_min'])/(row[f'ndmi_{season_name}_max'] - row[f'ndmi_{season_name}_min'] + 0.00001)

  dict[f'GCI_{season_name}_base'] = row[f'GCI_{season_name}_{base_year}']
  dict[f'GCI_{season_name}_base_norm'] = (row[f'GCI_{season_name}_{base_year}'] - row[f'GCI_{season_name}_min'])/(row[f'GCI_{season_name}_max'] - row[f'GCI_{season_name}_min'] + 0.00001)
  dict[f'GCI_{season_name}_target'] = row[f'GCI_{season_name}_{target_year}']
  dict[f'GCI_{season_name}_target_norm'] = (row[f'GCI_{season_name}_{target_year}'] - row[f'GCI_{season_name}_min'])/(row[f'GCI_{season_name}_max'] - row[f'GCI_{season_name}_min'] + 0.00001)
  df_append = pd.DataFrame([dict])
  df_main_ndvi = pd.concat([df_main_ndvi, df_append], ignore_index=True)

In [ ]:
df_main_ndvi[f'{season_name}_ndvi_diff'] = df_main_ndvi[f'ndvi_{season_name}_target'] - df_main_ndvi[f'ndvi_{season_name}_base']
df_main_ndvi[f'{season_name}_ndvi_diff_norm'] = df_main_ndvi[f'ndvi_{season_name}_target_norm'] - df_main_ndvi[f'ndvi_{season_name}_base_norm']

df_main_ndvi[f'{season_name}_ndmi_diff'] = df_main_ndvi[f'ndmi_{season_name}_target'] - df_main_ndvi[f'ndmi_{season_name}_base']
df_main_ndvi[f'{season_name}_ndmi_diff_norm'] = df_main_ndvi[f'ndmi_{season_name}_target_norm'] - df_main_ndvi[f'ndmi_{season_name}_base_norm']

df_main_ndvi[f'{season_name}_GCI_diff'] = df_main_ndvi[f'GCI_{season_name}_target'] - df_main_ndvi[f'GCI_{season_name}_base']
df_main_ndvi[f'{season_name}_GCI_diff_norm'] = df_main_ndvi[f'GCI_{season_name}_target_norm'] - df_main_ndvi[f'GCI_{season_name}_base_norm']

In [ ]:
df_main_ndvi.shape

(24000, 49)

##Saving progress

In [ ]:
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10/
df_main_ndvi.to_csv(f'DML_Allyears_dump_df_main_with_ndvi_ndmi_gci_{season_name}_{str_date}.csv', index=False)
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10/
df_ndvicalc = pd.read_csv(f'DML_Allyears_dump_df_main_with_ndvi_ndmi_gci_{season_name}_{str_date}.csv')
df_ndvicalc.head()

/content/drive/MyDrive/Sanya/DML/AEZ-10
/content/drive/MyDrive/Sanya/DML/AEZ-10


,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,.geo,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year,frequency,intensity,drought,ndvi_Kharif_target,ndvi_Kharif_target_norm,ndvi_Kharif_base,ndvi_Kharif_base_norm,GCI_Kharif_target,GCI_Kharif_target_norm,GCI_Kharif_base,GCI_Kharif_base_norm,ndmi_Kharif_base,ndmi_Kharif_base_norm,ndmi_Kharif_target,ndmi_Kharif_target_norm,Kharif_ndvi_diff,Kharif_ndvi_diff_norm,Kharif_ndmi_diff,Kharif_ndmi_diff_norm,Kharif_GCI_diff,Kharif_GCI_diff_norm
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2018,5.0,2.0,1.0,0.307611,0.861855,0.282017,0.704561,0.809972,0.827863,0.711542,0.612676,0.154264,0.999691,0.121871,0.000000,0.025594,0.157294,-0.032393,-0.999691,0.098430,0.215187
1,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2019,0.0,0.0,0.0,0.167375,0.000000,0.282017,0.704561,0.431295,0.000000,0.711542,0.612676,0.154264,0.999691,0.149823,0.862648,-0.114641,-0.704561,-0.004441,-0.137043,-0.280247,-0.612676
2,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2020,0.0,0.0,0.0,0.226731,0.364786,0.282017,0.704561,0.615263,0.402191,0.711542,0.612676,0.154264,0.999691,0.152251,0.937562,-0.055286,-0.339775,-0.002013,-0.062129,-0.096279,-0.210485
3,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2021,4.0,2.0,0.0,0.330079,0.999939,0.282017,0.704561,0.888700,0.999978,0.711542,0.612676,0.154264,0.999691,0.133397,0.355712,0.048062,0.295378,-0.020867,-0.643980,0.177158,0.387302
4,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279,2.0,2016,2018,5.0,2.0,1.0,0.296478,0.820886,0.300015,0.843412,0.790817,0.817366,0.801114,0.841534,0.140457,0.779565,0.105997,0.000000,-0.003537,-0.022526,-0.034460,-0.779565,-0.010297,-0.024167


#Adding Rainfall

In [ ]:
%cd /content/drive/MyDrive/Anika_Murarka/AEZ10/
df_prec_vf = pd.read_csv('AEZ10_Precipitation_data_watershed_lvl_global.csv').drop('Unnamed: 0', axis=1, errors='ignore')

/content/drive/MyDrive/Anika_Murarka/AEZ10


In [ ]:
df_prec_vf.head()

,objectid,wscode,Precipitation-2015-1,Precipitation-2015-2,Precipitation-2015-3,Precipitation-2015-4,Precipitation-2015-5,Precipitation-2015-6,Precipitation-2015-7,Precipitation-2015-8,Precipitation-2015-9,Precipitation-2015-10,Precipitation-2015-11,Precipitation-2015-12,Precipitation-2016-1,Precipitation-2016-2,Precipitation-2016-3,Precipitation-2016-4,Precipitation-2016-5,Precipitation-2016-6,Precipitation-2016-7,Precipitation-2016-8,Precipitation-2016-9,Precipitation-2016-10,Precipitation-2016-11,Precipitation-2016-12,Precipitation-2017-1,Precipitation-2017-2,Precipitation-2017-3,Precipitation-2017-4,Precipitation-2017-5,Precipitation-2017-6,Precipitation-2017-7,Precipitation-2017-8,Precipitation-2017-9,Precipitation-2017-10,Precipitation-2017-11,Precipitation-2017-12,Precipitation-2018-1,Precipitation-2018-2,Precipitation-2018-3,Precipitation-2018-4,Precipitation-2018-5,Precipitation-2018-6,Precipitation-2018-7,Precipitation-2018-8,Precipitation-2018-9,Precipitation-2018-10,Precipitation-2018-11,Precipitation-2018-12,Precipitation-2019-1,Precipitation-2019-2,Precipitation-2019-3,Precipitation-2019-4,Precipitation-2019-5,Precipitation-2019-6,Precipitation-2019-7,Precipitation-2019-8,Precipitation-2019-9,Precipitation-2019-10,Precipitation-2019-11,Precipitation-2019-12,Precipitation-2020-1,Precipitation-2020-2,Precipitation-2020-3,Precipitation-2020-4,Precipitation-2020-5,Precipitation-2020-6,Precipitation-2020-7,Precipitation-2020-8,Precipitation-2020-9,Precipitation-2020-10,Precipitation-2020-11,Precipitation-2020-12,Precipitation-2021-1,Precipitation-2021-2,Precipitation-2021-3,Precipitation-2021-4,Precipitation-2021-5,Precipitation-2021-6,Precipitation-2021-7,Precipitation-2021-8,Precipitation-2021-9,Precipitation-2021-10,Precipitation-2021-11,Precipitation-2021-12,Precipitation-2022-1,Precipitation-2022-2,Precipitation-2022-3,Precipitation-2022-4,Precipitation-2022-5,Precipitation-2022-6,Precipitation-2022-7,Precipitation-2022-8,Precipitation-2022-9,Precipitation-2022-10,Precipitation-2022-11,Precipitation-2022-12
0,292,3,5.244260,49.069224,57.226371,37.122832,17.780949,151.441303,148.838157,199.348842,39.382310,22.784577,0.118274,0.065450,3.849115,15.137682,35.273996,3.276447,24.444036,150.121912,496.876069,300.623550,79.692862,23.524755,0.000000,0.003633,2.127920,1.812915,0.397556,0.062777,22.560577,228.038524,345.447537,110.453108,158.556150,20.378741,0.033851,0.000000,0.000000,92.245460,4.270032,38.546556,27.360725,193.911188,431.344130,183.991329,27.498736,0.481666,0.570682,2.487173,6.803592,14.327260,8.667063,27.100542,1.694467,140.509511,241.652300,383.202913,426.154852,22.480956,1.373189,4.743672,3.746319,1.336670,41.548194,5.622008,53.459318,311.091538,162.714997,430.833630,117.804784,26.899210,0.358044,1.787538,3.157791,5.893648,18.785628,24.577581,64.544390,211.078094,133.283293,102.045115,172.033495,22.270173,0.627806,13.786110,9.401687,9.643146,2.766268,0.660894,2.321555,161.525970,445.545139,287.667774,150.731910,33.083731,0.0,2.582787
1,293,7,2.829607,35.291998,54.610287,19.940881,24.475408,137.864274,151.703552,280.494791,51.310614,13.039150,0.001012,0.038119,3.058454,6.980672,48.291122,7.669122,26.333584,178.948018,423.099724,398.487097,107.912235,22.950167,0.000000,0.000000,2.994577,2.873045,0.716994,0.543627,19.408439,172.818924,367.608842,154.645796,186.126678,14.233845,0.000000,0.004693,0.147844,68.357269,3.977177,30.864272,18.085249,181.363422,483.652434,195.728683,37.624482,0.753851,0.097773,6.168693,10.741888,10.365733,19.612565,32.649996,1.996092,145.778054,280.210934,339.832145,429.908457,17.415959,0.869889,5.009901,13.108062,8.106299,31.657502,11.467309,16.571433,287.626120,208.492445,392.633773,119.173034,34.103930,1.464743,1.750370,1.455558,9.081664,18.922696,13.882216,75.662174,182.805258,146.561121,103.566531,173.389458,9.719804,0.531764,21.126046,10.553346,11.280175,0.678726,1.697540,12.796263,130.242052,370.666321,315.887900,137.691149,35.697212,0.0,0.457104
2,294,31,8.681988,54.478778,6

In [ ]:
wsfile = pd.read_csv("AEZ10_all_points_with_ws_objectid.csv")
df_prec_vf = pd.merge(df_prec_vf, wsfile[['UID', 'objectid']], on='objectid', how='left')

In [ ]:
df_prec_vf

,objectid,wscode,Precipitation-2015-1,Precipitation-2015-2,Precipitation-2015-3,Precipitation-2015-4,Precipitation-2015-5,Precipitation-2015-6,Precipitation-2015-7,Precipitation-2015-8,Precipitation-2015-9,Precipitation-2015-10,Precipitation-2015-11,Precipitation-2015-12,Precipitation-2016-1,Precipitation-2016-2,Precipitation-2016-3,Precipitation-2016-4,Precipitation-2016-5,Precipitation-2016-6,Precipitation-2016-7,Precipitation-2016-8,Precipitation-2016-9,Precipitation-2016-10,Precipitation-2016-11,Precipitation-2016-12,Precipitation-2017-1,Precipitation-2017-2,Precipitation-2017-3,Precipitation-2017-4,Precipitation-2017-5,Precipitation-2017-6,Precipitation-2017-7,Precipitation-2017-8,Precipitation-2017-9,Precipitation-2017-10,Precipitation-2017-11,Precipitation-2017-12,Precipitation-2018-1,Precipitation-2018-2,Precipitation-2018-3,Precipitation-2018-4,Precipitation-2018-5,Precipitation-2018-6,Precipitation-2018-7,Precipitation-2018-8,Precipitation-2018-9,Precipitation-2018-10,Precipitation-2018-11,Precipitation-2018-12,Precipitation-2019-1,Precipitation-2019-2,Precipitation-2019-3,Precipitation-2019-4,Precipitation-2019-5,Precipitation-2019-6,Precipitation-2019-7,Precipitation-2019-8,Precipitation-2019-9,Precipitation-2019-10,Precipitation-2019-11,Precipitation-2019-12,Precipitation-2020-1,Precipitation-2020-2,Precipitation-2020-3,Precipitation-2020-4,Precipitation-2020-5,Precipitation-2020-6,Precipitation-2020-7,Precipitation-2020-8,Precipitation-2020-9,Precipitation-2020-10,Precipitation-2020-11,Precipitation-2020-12,Precipitation-2021-1,Precipitation-2021-2,Precipitation-2021-3,Precipitation-2021-4,Precipitation-2021-5,Precipitation-2021-6,Precipitation-2021-7,Precipitation-2021-8,Precipitation-2021-9,Precipitation-2021-10,Precipitation-2021-11,Precipitation-2021-12,Precipitation-2022-1,Precipitation-2022-2,Precipitation-2022-3,Precipitation-2022-4,Precipitation-2022-5,Precipitation-2022-6,Precipitation-2022-7,Precipitation-2022-8,Precipitation-2022-9,Precipitation-2022-10,Precipitation-2022-11,Precipitation-2022-12,UID
0,292,3,5.244260,49.069224,57.226371,37.122832,17.780949,151.441303,148.838157,199.348842,39.382310,22.784577,0.118274,0.065450,3.849115,15.137682,35.273996,3.276447,24.444036,150.121912,496.876069,300.623550,79.692862,23.524755,0.0,0.003633,2.127920,1.812915,0.397556,0.062777,22.560577,228.038524,345.447537,110.453108,158.556150,20.378741,0.033851,0.000000,0.0,92.245460,4.270032,38.546556,27.360725,193.911188,431.344130,183.991329,27.498736,0.481666,0.570682,2.487173,6.803592,14.327260,8.667063,27.100542,1.694467,140.509511,241.652300,383.202913,426.154852,22.480956,1.373189,4.743672,3.746319,1.336670,41.548194,5.622008,53.459318,311.091538,162.714997,430.833630,117.804784,26.899210,0.358044,1.787538,3.157791,5.893648,18.785628,24.577581,64.544390,211.078094,133.283293,102.045115,172.033495,22.270173,0.627806,13.786110,9.401687,9.643146,2.766268,0.660894,2.321555,161.525970,445.545139,287.667774,150.731910,33.083731,0.0,2.582787,5544.0
1,292,3,5.244260,49.069224,57.226371,37.122832,17.780949,151.441303,148.838157,199.348842,39.382310,22.784577,0.118274,0.065450,3.849115,15.137682,35.273996,3.276447,24.444036,150.121912,496.876069,300.623550,79.692862,23.524755,0.0,0.003633,2.127920,1.812915,0.397556,0.062777,22.560577,228.038524,345.447537,110.453108,158.556150,20.378741,0.033851,0.000000,0.0,92.245460,4.270032,38.546556,27.360725,193.911188,431.344130,183.991329,27.498736,0.481666,0.570682,2.487173,6.803592,14.327260,8.667063,27.100542,1.694467,140.509511,241.652300,383.202913,426.154852,22.480956,1.373189,4.743672,3.746319,1.336670,41.548194,5.622008,53.459318,311.091538,162.714997,430.833630,117.804784,26.899210,0.358044,1.787538,3.157791,5.893648,18.785628,24.577581,64.544390,211.078094,133.283293,102.045115,172.033495,22.270173,0.627806,13.786110,9.401687,9.643146,2.766268,0.660894,2.321555,161.525970,445.545139,287.667774,150.731910,33.083731,0.0,2.582787,5574.0
2,292,3,5.244260,49.069224,57.226371,

In [ ]:
df_prec_vf_new = df_prec_vf[['UID']]

if season_name == "Kharif":
    for year in range(2015, 2022):
        df_prec_vf_new[f'precipitation_Kharif_{year}'] = df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(7)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(8)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(9)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(10)]
elif season_name == "Rabi":
    for year in range(2015, 2022):
        df_prec_vf_new[f'precipitation_Kharif_{year}'] = df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(7)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(8)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(9)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(10)]
        df_prec_vf_new['precipitation_Rabi_' + str(year)] = df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(11)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(12)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(1)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(2)]
elif season_name == "Zaid":
    for year in range(2015, 2022):
        df_prec_vf_new[f'precipitation_Kharif_{year}'] = df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(7)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(8)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(9)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(10)]
        df_prec_vf_new['precipitation_Rabi_' + str(year)] = df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(11)] + df_prec_vf['Precipitation' + '-' + str(year) + '-' + str(12)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(1)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(2)]
        df_prec_vf_new['precipitation_Zaid_' + str(year)] = df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(3)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(4)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(5)] + df_prec_vf['Precipitation' + '-' + str(year+1) + '-' + str(6)]

df_ndvicalc = pd.merge(df_ndvicalc, df_prec_vf_new, on='UID')

In [ ]:
df_ndvicalc

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,.geo,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year,frequency,intensity,drought,ndvi_Kharif_target,ndvi_Kharif_target_norm,ndvi_Kharif_base,ndvi_Kharif_base_norm,GCI_Kharif_target,GCI_Kharif_target_norm,GCI_Kharif_base,GCI_Kharif_base_norm,ndmi_Kharif_base,ndmi_Kharif_base_norm,ndmi_Kharif_target,ndmi_Kharif_target_norm,Kharif_ndvi_diff,Kharif_ndvi_diff_norm,Kharif_ndmi_diff,Kharif_ndmi_diff_norm,Kharif_GCI_diff,Kharif_GCI_diff_norm,precipitation_Kharif_2015,precipitation_Kharif_2016,precipitation_Kharif_2017,precipitation_Kharif_2018,precipitation_Kharif_2019,precipitation_Kharif_2020,precipitation_Kharif_2021
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2018,5.0,2.0,1.0,0.307611,0.861855,0.282017,0.704561,0.809972,0.827863,0.711542,0.612676,0.154264,0.999691,0.121871,0.000000,0.025594,0.157294,-0.032393,-0.999691,0.098430,0.215187,557.600311,890.564727,666.522163,834.532886,772.737533,728.041227,704.453280
1,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2019,0.0,0.0,0.0,0.167375,0.000000,0.282017,0.704561,0.431295,0.000000,0.711542,0.612676,0.154264,0.999691,0.149823,0.862648,-0.114641,-0.704561,-0.004441,-0.137043,-0.280247,-0.612676,557.600311,890.564727,666.522163,834.532886,772.737533,728.041227,704.453280
2,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2020,0.0,0.0,0.0,0.226731,0.364786,0.282017,0.704561,0.615263,0.402191,0.711542,0.612676,0.154264,0.999691,0.152251,0.937562,-0.055286,-0.339775,-0.002013,-0.062129,-0.096279,-0.210485,557.600311,890.564727,666.522163,834.532886,772.737533,728.041227,704.453280
3,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2021,4.0,2.0,0.0,0.330079,0.999939,0.282017,0.704561,0.888700,0.999978,0.711542,0.612676,0.154264,0.999691,0.133397,0.355712,0.048062,0.295378,-0.020867,-0.643980,0.177158,0.387302,557.600311,890.564727,666.522163,834.532886,772.737533,728.041227,704.453280
4,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279,2.0,2016,2018,5.0,2.0,1.0,0.296478,0.820886,0.300015,0.843412,0.790817,0.817366,0.801114,0.841534,0.140457,0.779565,0.105997,0.000000,-0.003537,-0.022526,-0.034460,-0.779565,-0.010297,-0.024167,557.600311,890.564727,666.522163,834.532886,772.737533,728.041227,704.453280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27307,7997,0,control,BHAINSDEHI,BETUL,MADHYA PRADESH,0.0,0,0.000000,1407.631167,616.318350,460.077597,56.069766,718.796782,1,21.693101,77.680871,42.331378,3.206368,"{""type"":""Point"",""coordinates"":[77.68

##Creating observations with rainfall

In [ ]:
df_prec = df_ndvicalc

In [ ]:
df_prec[f'precip_baseyr_{season_name}'] = 0
df_prec[f'precip_targetyr_{season_name}'] = 0
for index, row in df_prec.iterrows():
    base_year = row['base_year']
    target_year = row['target_year']
    df_prec.at[index, f'precip_baseyr_{season_name}'] = row[f'precipitation_{season_name}_{base_year}']
    df_prec.at[index, f'precip_targetyr_{season_name}'] = row[f'precipitation_{season_name}_{target_year}']
    if season_name == "Rabi" or season_name == "Zaid":
        df_prec.at[index, f'precip_baseyr_Kharif'] = row[f'precipitation_Kharif_{base_year}']
        df_prec.at[index, f'precip_targetyr_Kharif'] = row[f'precipitation_Kharif_{target_year}']
    if season_name == "Zaid":
        df_prec.at[index, f'precip_baseyr_Rabi'] = row[f'precipitation_Rabi_{base_year}']
        df_prec.at[index, f'precip_targetyr_Rabi'] = row[f'precipitation_Rabi_{target_year}']
seasons = ["Kharif", "Rabi", "Zaid"]
for season in seasons:
    df_prec = df_prec.drop([f'precipitation_{season}_2015',f'precipitation_{season}_2016',f'precipitation_{season}_2017',f'precipitation_{season}_2018',f'precipitation_{season}_2019',f'precipitation_{season}_2020',f'precipitation_{season}_2021',], axis=1, errors='ignore')
df_prec.shape

(27312, 51)

##Saving progress

In [ ]:
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10
df_prec.to_csv(f"DML_Allyears_NDVI_df_{season_name}_{str_date}.csv", index=False)
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10
df_main = pd.read_csv(f"DML_Allyears_NDVI_df_{season_name}_{str_date}.csv")
print((df_main.columns))

/content/drive/MyDrive/Sanya/DML/AEZ-10
/content/drive/MyDrive/Sanya/DML/AEZ-10
Index(['UID', 'Asset ID', 'Asset Name', 'Block', 'District', 'State',
       'Total_Expenditure', 'Treatment', 'dist_closest_crop',
       'dist_closest_lin', 'dist_closest_river', 'dist_closest_road',
       'dist_closest_upstream_forest', 'elevation', 'flow_accumulation', 'lat',
       'lon', 'proximity_water', 'slope', '.geo', 'HSG', 'pH', 'OC', 'CEC',
       'drainage_density', 'recent_cropping_intensity', 'base_year',
       'target_year', 'frequency', 'intensity', 'drought',
       'ndvi_Kharif_target', 'ndvi_Kharif_target_norm', 'ndvi_Kharif_base',
       'ndvi_Kharif_base_norm', 'GCI_Kharif_target', 'GCI_Kharif_target_norm',
       'GCI_Kharif_base', 'GCI_Kharif_base_norm', 'ndmi_Kharif_base',
       'ndmi_Kharif_base_norm', 'ndmi_Kharif_target',
       'ndmi_Kharif_target_norm', 'Kharif_ndvi_diff', 'Kharif_ndvi_diff_norm',
       'Kharif_ndmi_diff', 'Kharif_ndmi_diff_norm', 'Kharif_GCI_diff',
     

#One-hot encoding

In [ ]:
df_psm = df_main.copy()

#creating instance of one-hot-encoder
encoder = OneHotEncoder(handle_unknown='ignore')

#perform one-hot encoding
encoder_df = pd.DataFrame(encoder.fit_transform( df_main[['HSG']]).toarray())

#merge one-hot encoded columns back with original DataFrame
df_main =  df_main.join(encoder_df)

df_main =  df_main.rename(columns={0:'HSG_3',1:'HSG_4', 2:'HSG_13', 3:'HSG_14'})
df_main.head()

,UID,Asset ID,Asset Name,Block,District,State,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,.geo,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year,frequency,intensity,drought,ndvi_Kharif_target,ndvi_Kharif_target_norm,ndvi_Kharif_base,ndvi_Kharif_base_norm,GCI_Kharif_target,GCI_Kharif_target_norm,GCI_Kharif_base,GCI_Kharif_base_norm,ndmi_Kharif_base,ndmi_Kharif_base_norm,ndmi_Kharif_target,ndmi_Kharif_target_norm,Kharif_ndvi_diff,Kharif_ndvi_diff_norm,Kharif_ndmi_diff,Kharif_ndmi_diff_norm,Kharif_GCI_diff,Kharif_GCI_diff_norm,precip_baseyr_Kharif,precip_targetyr_Kharif,HSG_3,HSG_4,HSG_13,HSG_14,4
0,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2018,5.0,2.0,1.0,0.307611,0.861855,0.282017,0.704561,0.809972,0.827863,0.711542,0.612676,0.154264,0.999691,0.121871,0.000000,0.025594,0.157294,-0.032393,-0.999691,0.098430,0.215187,890.564727,834.532886,1.0,0.0,0.0,0.0,0.0
1,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2019,0.0,0.0,0.0,0.167375,0.000000,0.282017,0.704561,0.431295,0.000000,0.711542,0.612676,0.154264,0.999691,0.149823,0.862648,-0.114641,-0.704561,-0.004441,-0.137043,-0.280247,-0.612676,890.564727,772.737533,1.0,0.0,0.0,0.0,0.0
2,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2020,0.0,0.0,0.0,0.226731,0.364786,0.282017,0.704561,0.615263,0.402191,0.711542,0.612676,0.154264,0.999691,0.152251,0.937562,-0.055286,-0.339775,-0.002013,-0.062129,-0.096279,-0.210485,890.564727,728.041227,1.0,0.0,0.0,0.0,0.0
3,1,33001475173,Dabri nirman kary,Chhuria,RAJNANDAGON,CHHATTISGARH,124872.0,1,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1,20.948678,80.454504,0.604106,3.419834,"{""type"":""Point"",""coordinates"":[80.454504,20.94...",3.0,6.3,0.53,6.0,0.005974,1.0,2016,2021,4.0,2.0,0.0,0.330079,0.999939,0.282017,0.704561,0.888700,0.999978,0.711542,0.612676,0.154264,0.999691,0.133397,0.355712,0.048062,0.295378,-0.020867,-0.643980,0.177158,0.387302,890.564727,704.453280,1.0,0.0,0.0,0.0,0.0
4,2,33001544850,dabri nirman,Dongargarh,RAJNANDAGON,CHHATTISGARH,98384.0,1,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1,21.114584,80.454499,0.736070,3.867191,"{""type"":""Point"",""coordinates"":[80.454499,21.11...",3.0,6.3,0.53,6.0,0.006279,2.0,2016,2018,5.0,2.0,1.0,0.296478,0.820886,0.300015,0.843412,0.790817,0.817366,0.801114,0.841534,0.140457,0.779565,0.105997,0.000000,-0.003537,-0.022526,-0.034460,-0.779565,-0.010297,-0.024167,890.564727,834.532886,1.0,0.0,0.0,0.0,0.0


In [ ]:
df_main = df_main.drop(['4', '.geo', 'size', 'State' 'Total_Expenditure', 'pointInRegion', 'Asset Name', 'Block', 'District', 'Panchayat', 'block_coords', 'district_coords', 'state_coords', 'Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2', 'Unnamed: 0.3'], axis=1, errors='ignore')


In [ ]:
df_main.drop('State', axis=1, inplace=True)

In [ ]:
def clean_dataset(df):
    assert isinstance(df, pd.DataFrame), "df needs to be a pd.DataFrame"
    df.dropna(inplace=True)
    indices_to_keep = ~df.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    return df[indices_to_keep].astype(np.float64)

df_clean = clean_dataset(df_main)

print(df_clean.shape, df_clean[df_clean['Treatment'] == 1.0].shape, df_clean[df_clean['Treatment'] == 0.0].shape)

(26128, 51) (5636, 51) (20492, 51)


In [ ]:
df_clean

,UID,Asset ID,Total_Expenditure,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year,frequency,intensity,drought,ndvi_Kharif_target,ndvi_Kharif_target_norm,ndvi_Kharif_base,ndvi_Kharif_base_norm,GCI_Kharif_target,GCI_Kharif_target_norm,GCI_Kharif_base,GCI_Kharif_base_norm,ndmi_Kharif_base,ndmi_Kharif_base_norm,ndmi_Kharif_target,ndmi_Kharif_target_norm,Kharif_ndvi_diff,Kharif_ndvi_diff_norm,Kharif_ndmi_diff,Kharif_ndmi_diff_norm,Kharif_GCI_diff,Kharif_GCI_diff_norm,precip_baseyr_Kharif,precip_targetyr_Kharif,HSG_3,HSG_4,HSG_13,HSG_14,4
0,1.0,3.300148e+10,124872.0,1.0,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1.0,20.948678,80.454504,0.604106,3.419834,3.0,6.3,0.53,6.0,0.005974,1.0,2016.0,2018.0,5.0,2.0,1.0,0.307611,0.861855,0.282017,0.704561,0.809972,0.827863,0.711542,0.612676,0.154264,0.999691,0.121871,0.000000,0.025594,0.157294,-0.032393,-0.999691,0.098430,0.215187,890.564727,834.532886,1.0,0.0,0.0,0.0,0.0
1,1.0,3.300148e+10,124872.0,1.0,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1.0,20.948678,80.454504,0.604106,3.419834,3.0,6.3,0.53,6.0,0.005974,1.0,2016.0,2019.0,0.0,0.0,0.0,0.167375,0.000000,0.282017,0.704561,0.431295,0.000000,0.711542,0.612676,0.154264,0.999691,0.149823,0.862648,-0.114641,-0.704561,-0.004441,-0.137043,-0.280247,-0.612676,890.564727,772.737533,1.0,0.0,0.0,0.0,0.0
2,1.0,3.300148e+10,124872.0,1.0,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1.0,20.948678,80.454504,0.604106,3.419834,3.0,6.3,0.53,6.0,0.005974,1.0,2016.0,2020.0,0.0,0.0,0.0,0.226731,0.364786,0.282017,0.704561,0.615263,0.402191,0.711542,0.612676,0.154264,0.999691,0.152251,0.937562,-0.055286,-0.339775,-0.002013,-0.062129,-0.096279,-0.210485,890.564727,728.041227,1.0,0.0,0.0,0.0,0.0
3,1.0,3.300148e+10,124872.0,1.0,501.264597,550.223539,344.301082,6330.266231,9.639730,374.130891,1.0,20.948678,80.454504,0.604106,3.419834,3.0,6.3,0.53,6.0,0.005974,1.0,2016.0,2021.0,4.0,2.0,0.0,0.330079,0.999939,0.282017,0.704561,0.888700,0.999978,0.711542,0.612676,0.154264,0.999691,0.133397,0.355712,0.048062,0.295378,-0.020867,-0.643980,0.177158,0.387302,890.564727,704.453280,1.0,0.0,0.0,0.0,0.0
4,2.0,3.300154e+10,98384.0,1.0,176.865251,981.439779,2573.537992,3222.104076,51.883314,349.520972,1.0,21.114584,80.454499,0.736070,3.867191,3.0,6.3,0.53,6.0,0.006279,2.0,2016.0,2018.0,5.0,2.0,1.0,0.296478,0.820886,0.300015,0.843412,0.790817,0.817366,0.801114,0.841534,0.140457,0.779565,0.105997,0.000000,-0.003537,-0.022526,-0.034460,-0.779565,-0.010297,-0.024167,890.564727,834.532886,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27307,7997.0,0.000000e+00,0.0,0.0,0.000000,1407.631167,616.318350,460.077597,56.069766,718.796782,1.0,21.693101,77.680871,42.331378,3.206368,4.0,7.1,0.97,36.0,0.005844,1.0,2016.0,2021.0,8.0,2.0,1.0,0.302189,0.999935,0.147484,0.000000,0.783144,0.999977,0.348399,0.000000,0.091272,0.000000,0.146545,0.999819,0.154705,0.999935,0.055273,0.999819,0.434745,0.999977,744.582726,631.436767,0.0,1.0,0.0,0.0,0.0
27308,7998.0,0.000000e+00,0.0,0.0,0.000000,1355.883137,464.989212,1484.229627,21.502521,659.955370,1.0,21.682861,77.752018,0.093842,2.865304,4.0,7.1,0.97,36.0,0.005772,2.0,2016.0,2018.0,4.0,2.0,0.0,0.256876,0.866955,0.181603,0.000000,0.763797,0.999966,0.467339,0.000000,0.098609,0.308775,0.098966,0.315754,0.075274,0.866955,0.000357,0.006979,0.296458,0.999966,744.582726,554.749693,0.0,1.0,0.0,0.0,0.0
27309,7998.0,0.000000e+00,0.0,0.0,0.000000,1355.883137,464.989212,1484.229627,21.502521,659.955370,1.0,21.682861,77.752018,0.093842,2.865304,4.0,7.1,0.97,36.0,0.005772,2.0,2016.0,2019.0,0.0,0.0,0.0,0.201763,0.232197,0.181603,0.000000

In [ ]:
# RUN THIS ONLY if '4' shows up at the end: df_clean = df_clean.iloc[:, :-1]

In [ ]:
df_clean.drop('Total_Expenditure', axis=1, inplace=True)
df_clean.drop('Asset ID', axis=1, inplace=True)

#Removing 20% reserved valid farm ponds from prepared data

1. For each UID in df_reserved_data, find all matching rows in df_clean.
2. Store these matching rows in a new DataFrame data_reserved_final.
3. Remove these matching rows from df_clean.

In [ ]:
# Step 1: Create a boolean mask to check if UIDs in df_reserved_data exist in df_clean
mask = df_reserved_data['UID'].isin(df_clean['UID'])

# Step 2: Count how many UIDs from df_reserved_data are in df_clean
matching_uid_count = mask.sum()

# Step 3: Print the result
print(f"Number of UIDs in df_reserved_data that exist in df_clean: {matching_uid_count}")

Number of UIDs in df_reserved_data that exist in df_clean: 280


In [ ]:
import random

# Step 1: Select a random UID from df_reserved_data
random_uid_reserved = random.choice(df_reserved_data['UID'].tolist())
print(f"Random UID selected from df_reserved_data: {random_uid_reserved}")

# Step 2: Count occurrences of the random UID in df_clean
count_in_df_clean = df_clean[df_clean['UID'] == random_uid_reserved].shape[0]

# Step 3: Print the count
print(f"Number of occurrences of UID = {random_uid_reserved} in df_clean: {count_in_df_clean}")

Random UID selected from df_reserved_data: 351
Number of occurrences of UID = 351 in df_clean: 4


In [ ]:
df_clean.shape

(26128, 49)

In [ ]:
df_reserved_data.shape

(300, 26)

In [ ]:
# Step 1: Initialize an empty DataFrame for storing the matches
data_reserved_final = pd.DataFrame()

# Step 2: Iterate over each UID in df_reserved_data
for uid in df_reserved_data['UID'].unique():
    # Find all matches in df_clean
    matches = df_clean[df_clean['UID'] == uid]

    # Append the matches to data_reserved_final
    data_reserved_final = pd.concat([data_reserved_final, matches], axis=0)

    # Remove the matches from df_clean
    df_clean = df_clean[df_clean['UID'] != uid]

# Step 3: Reset index of the final DataFrame if needed
data_reserved_final = data_reserved_final.reset_index(drop=True)

# Step 4: Print or inspect the resulting DataFrames
print(f"Data reserved final shape: {data_reserved_final.shape}")
print(f"Data clean shape after removal: {df_clean.shape}")

Data reserved final shape: (1120, 49)
Data clean shape after removal: (25008, 49)


#PSM Matching, Saving the dataframe

In [ ]:
#PSM Matching is a different concept. Here we are only seeing the ratio of control:treated is correct.
#Seeding to ensure values are consistent across diff runs
#We need not do this for testing

if not psm_matched:
    df_clean = df_clean.drop(4, axis = 1, errors='ignore')
    print(df_clean.shape, df_clean[df_clean['Treatment'] == 1.0].shape, df_clean[df_clean['Treatment'] == 0.0].shape)
    treated_count = df_clean[df_clean['Treatment'] == 1.0].shape[0]
    control_count = df_clean[df_clean['Treatment'] == 0.0].shape[0]
    req_fr = (3.0 * treated_count / (1 + control_count))
    print(req_fr)

if not psm_matched:
    # Resample data points. Number of control points = 3*number of treated points
    df_sample_treated = df_clean[df_clean['Treatment'] == 1]
    df_sample_control = df_clean[df_clean['Treatment'] == 0]
    df_sample_treated = df_sample_treated.sort_values('UID')
    df_sample_control = df_sample_control.sort_values('UID')

    df_sample_control = df_sample_control.sample(frac=req_fr, random_state=153) # num of control = 3*number of treated points
    df_sampled = pd.concat([df_sample_treated, df_sample_control], axis=0, ignore_index=True)
    df_sampled = df_sampled.sample(frac=1, random_state=7359) # to shuffle the rows
    df_sampled.shape

    print(df_sampled.shape, df_sampled[df_sampled['Treatment'] == 1.0].shape, df_sampled[df_sampled['Treatment'] == 0.0].shape)

(25008, 48) (4516, 48) (20492, 48)
0.6611037915385741
(18063, 48) (4516, 48) (13547, 48)


We can retain drought... it helps in HTE

In [ ]:
df_sampled.head()

,UID,Treatment,dist_closest_crop,dist_closest_lin,dist_closest_river,dist_closest_road,dist_closest_upstream_forest,elevation,flow_accumulation,lat,lon,proximity_water,slope,HSG,pH,OC,CEC,drainage_density,recent_cropping_intensity,base_year,target_year,frequency,intensity,drought,ndvi_Kharif_target,ndvi_Kharif_target_norm,ndvi_Kharif_base,ndvi_Kharif_base_norm,GCI_Kharif_target,GCI_Kharif_target_norm,GCI_Kharif_base,GCI_Kharif_base_norm,ndmi_Kharif_base,ndmi_Kharif_base_norm,ndmi_Kharif_target,ndmi_Kharif_target_norm,Kharif_ndvi_diff,Kharif_ndvi_diff_norm,Kharif_ndmi_diff,Kharif_ndmi_diff_norm,Kharif_GCI_diff,Kharif_GCI_diff_norm,precip_baseyr_Kharif,precip_targetyr_Kharif,HSG_3,HSG_4,HSG_13,HSG_14
5378,7496.0,0.0,0.000000,287.759956,216.593951,1060.696793,158.776318,660.221945,51.0,22.412382,79.338532,29.287390,3.726361,4.0,7.9,0.75,43.0,0.005431,1.0,2016.0,2020.0,0.0,0.0,0.0,0.132758,0.000000,0.198876,0.482584,0.342409,0.000000,0.533682,0.494642,0.090552,0.182167,0.200114,0.999925,-0.066118,-0.482584,0.109563,0.817758,-0.191273,-0.494642,900.717236,738.252621,0.0,1.0,0.0,0.0
17332,6347.0,0.0,0.000000,7119.887440,2045.068949,152.211052,40.765110,651.949635,4.0,21.920555,77.891616,0.387097,2.534660,4.0,7.1,0.97,36.0,0.005622,1.0,2016.0,2020.0,0.0,0.0,0.0,0.237473,0.981987,0.152323,0.000000,0.635642,0.992620,0.364123,0.000000,0.093715,0.021939,0.123350,0.998570,0.085150,0.981987,0.029635,0.976631,0.271520,0.992620,945.095583,827.788029,0.0,1.0,0.0,0.0
13048,5975.0,0.0,2468.805354,193.427424,363.375735,3658.714120,38.446053,550.776055,1.0,21.826771,79.446599,0.000000,3.603385,4.0,7.6,1.40,0.0,0.005428,3.0,2016.0,2018.0,4.0,2.0,0.0,0.357351,0.999922,0.228408,0.000000,1.030010,0.999975,0.626044,0.000000,0.122594,0.000000,0.131175,0.211844,0.128943,0.999922,0.008581,0.211844,0.403966,0.999975,848.437612,797.405811,0.0,1.0,0.0,0.0
2510,808.0,1.0,0.000000,3364.657809,525.888333,1784.342229,105.291939,494.171709,2.0,22.999089,76.753512,1.829912,3.266721,14.0,7.9,0.75,43.0,0.004273,2.0,2016.0,2020.0,0.0,0.0,0.0,0.098430,0.000000,0.161002,0.465147,0.236398,0.000000,0.442565,0.479589,0.046070,0.231560,0.166143,0.999936,-0.062572,-0.465147,0.120072,0.768376,-0.206167,-0.479589,1173.287919,717.155156,0.0,0.0,0.0,1.0
9748,7855.0,0.0,0.000000,3832.923097,2398.224330,463.493819,136.435807,352.766849,57.0,24.203713,81.108842,15.651026,2.872150,4.0,7.9,0.75,43.0,0.005487,2.0,2016.0,2021.0,4.0,2.0,0.0,0.181538,0.935621,0.105657,0.165923,0.439370,0.999958,0.255440,0.228021,0.072777,0.224783,0.159586,0.999911,0.075881,0.769698,0.086809,0.775128,0.183930,0.771937,1034.920241,527.190319,0.0,1.0,0.0,0.0


In [ ]:
# export DF
%cd /content/drive/MyDrive/Sanya/DML/AEZ-10
if psm_matched:
    df_sampled.to_csv(f"DML_Allyears_NDVI_df_{season_name}_{str_date}_psm_matched.csv", index=False)
else:
    df_sampled.to_csv(f"DML_Allyears_NDVI_df_{season_name}_{str_date}_random.csv", index=False)

/content/drive/MyDrive/Sanya/DML/AEZ-10


In [ ]:
#export reserved testing points
data_reserved_final.to_csv(f"reserved_{season_name}_{str_date}.csv", index=False)